# Margin of Error — Data Exploration

This notebook demonstrates key analysis features.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').parent))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats


## 1. Load Sample Data


In [ ]:
from backend.ingestion.aec_loader import load_election_config
from backend.ingestion.csv_loader import (
    load_candidates, load_booths, load_live_count,
    load_declaration_votes, load_historical_booths
)

DATA_DIR = Path('data/sample')

election = load_election_config(str(DATA_DIR / 'election_config.json'))
print(f'Election: {election.electorate}, {election.state}')
print(f'Date: {election.election_date}')
print(f'Enrolled: {election.enrolled_voters:,}')
print(f'TCP: {election.tcp_candidates}')


In [ ]:
live_df = load_live_count(str(DATA_DIR / 'live_count.csv'))
print(f'Live count shape: {live_df.shape}')
live_df.head()


## 2. TCP Margin Calculation


In [ ]:
from backend.analysis.live_count import (
    calculate_tcp_margin, calculate_count_progress,
    count_status, recount_risk
)

margin, pct = calculate_tcp_margin(live_df)
print(f'TCP Margin: {margin:+,} votes')
print(f'TCP Margin: {pct:+.2f}pp from 50%')

primary_cols = [c for c in live_df.columns if '_primary' in c]
total_counted = int(live_df[primary_cols].sum().sum())
enrolled = election.enrolled_voters

counted_pct = calculate_count_progress(total_counted, enrolled)
print(f'Count progress: {counted_pct:.1f}%')
print(f'Status: {count_status(counted_pct)}')
print(f'Recount risk: {recount_risk(margin)[0]}')


## 3. Preference Flow Modelling


In [ ]:
from backend.analysis.preferences import PreferenceModel

model = PreferenceModel(election.candidates)
flows = model.default_pref_flows()
print('Default preference flows:')
for party, dests in flows.items():
    print(f'  {party}: ALP {dests["ALP"]*100:.0f}% | LIB {dests["LIB"]*100:.0f}%')

primary_votes = {'ALP': 38.0, 'LIB': 35.0, 'GRN': 14.0, 'IND': 10.0, 'informal': 3.0}
tcp = model.flow_to_tcp(primary_votes, flows)
print(f'\nProjected TCP: ALP {tcp["ALP"]:.2f}% | LIB {tcp["LIB"]:.2f}%')


## 4. Monte Carlo Simulation


In [ ]:
from backend.analysis.monte_carlo import MonteCarloEngine

engine = MonteCarloEngine(base_margin=margin, std_dev=1500)
results = engine.run_simulation(n_iterations=10000)

alp_prob = engine.probability_of_victory(results, 'ALP')
ci = engine.confidence_interval(results, 0.95)

print(f'P(ALP wins): {alp_prob:.1%}')
print(f'95% CI: [{ci[0]:+,}, {ci[1]:+,}]')

# Plot
x = np.linspace(results.min(), results.max(), 300)
y = stats.norm.pdf(x, results.mean(), results.std())

fig = go.Figure()
fig.add_trace(go.Scatter(x=x[x > 0], y=y[x > 0], fill='tozeroy',
                          fillcolor='rgba(229,57,53,0.2)', line_color='#E53935', name='ALP Win'))
fig.add_trace(go.Scatter(x=x[x <= 0], y=y[x <= 0], fill='tozeroy',
                          fillcolor='rgba(21,101,192,0.2)', line_color='#1565C0', name='LIB Win'))
fig.add_vline(x=margin, line_dash='dash', annotation_text=f'Projected: {margin:+,}')
fig.update_layout(title='Monte Carlo Margin Distribution', height=400)
fig.show()


## 5. Swing Analysis


In [ ]:
from backend.analysis.swing import calculate_primary_swing, apply_swing_scenario

# Compare to 2022
prev_alp = 35.8
curr_alp = 38.0
swing = calculate_primary_swing(curr_alp, prev_alp)
print(f'ALP primary swing: {swing:+.1f}pp')

# Scenario
base = {'ALP': 38.0, 'LIB': 35.0, 'GRN': 14.0, 'IND': 10.0}
scenario_swings = {'ALP': 2.0, 'LIB': -1.5, 'GRN': 0.5, 'IND': -1.0}
adjusted = apply_swing_scenario(base, scenario_swings)
print('\nScenario adjusted votes:')
for p, v in adjusted.items():
    print(f'  {p}: {v:.1f}%')


## 6. Poll Aggregation


In [ ]:
from backend.analysis.poll_aggregator import PollAggregator
from datetime import date, timedelta

agg = PollAggregator()

# Add some sample polls
today = date.today()
polls = [
    {'date': str(today - timedelta(days=5)), 'pollster': 'YouGov', 'alp': 38.5, 'lib': 34.5, 'grn': 14.0, 'ind': 10.0, 'others': 3.0},
    {'date': str(today - timedelta(days=12)), 'pollster': 'Essential', 'alp': 37.0, 'lib': 36.0, 'grn': 13.5, 'ind': 10.5, 'others': 3.0},
    {'date': str(today - timedelta(days=20)), 'pollster': 'Newspoll', 'alp': 37.5, 'lib': 35.5, 'grn': 14.0, 'ind': 10.0, 'others': 3.0},
]
for p in polls:
    agg.add_poll(p)

aggregated = agg.weighted_average(half_life_days=14)
print('Aggregated primary votes (14-day half-life):')
for party, pct in aggregated.items():
    print(f'  {party}: {pct:.1f}%')
